# Short Notebook 1 - Two-Stage LightGBM Model

**Team:** [35] ∫ √(tan x) dx

**Members:** Antonije Mirkovic (115014), ...

---

This notebook implements a two-stage forecasting approach:
1. **Stage 1:** LightGBM classifier predicts delivery probability
2. **Stage 2:** LightGBM quantile regressor (α=0.10) predicts delivery weight

Final prediction = regressor_output × classifier_probability

## Imports

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from datetime import timedelta

## Load Data

In [2]:
receivals = pd.read_csv('./Project_materials/data/kernel/receivals.csv')
purchase_orders = pd.read_csv('./Project_materials/data/kernel/purchase_orders.csv')
prediction_mapping = pd.read_csv('./Project_materials/data/prediction_mapping.csv')
sample_submission = pd.read_csv('./Project_materials/data/sample_submission.csv')

# Convert dates
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)
purchase_orders['delivery_date'] = pd.to_datetime(purchase_orders['delivery_date'], utc=True).dt.tz_localize(None)
purchase_orders['created_date_time'] = pd.to_datetime(purchase_orders['created_date_time'], utc=True).dt.tz_localize(None)
prediction_mapping['forecast_start_date'] = pd.to_datetime(prediction_mapping['forecast_start_date'])
prediction_mapping['forecast_end_date'] = pd.to_datetime(prediction_mapping['forecast_end_date'])

## Data Cleaning

In [3]:
receivals = receivals[receivals['net_weight'] > 0]
receivals = receivals[receivals['rm_id'].notna()]
receivals = receivals.sort_values('date_arrival')
print(f"Clean receivals: {len(receivals)}")

Clean receivals: 122383


## Create Training Data

Generate training samples with multiple forecast horizons from 2024 data.

In [4]:
train_dates = pd.date_range(start='2024-01-01', end='2024-11-30', freq='MS')
forecast_horizons = [7, 30, 60, 90, 150]

print(f"Using {len(train_dates)} training dates x {len(forecast_horizons)} horizons")

training_data = []
active_rm_ids = receivals[receivals['date_arrival'] >= '2024-01-01']['rm_id'].unique()
print(f"Active rm_ids in 2024: {len(active_rm_ids)}")

for i, train_date in enumerate(train_dates):
    print(f"Processing date {i+1}/{len(train_dates)}: {train_date.date()}...")
    
    for rm_id in active_rm_ids:
        hist = receivals[
            (receivals['rm_id'] == rm_id) &
            (receivals['date_arrival'] < train_date)
        ]
        
        if len(hist) == 0:
            continue
        
        cutoff_365 = train_date - timedelta(days=365)
        cutoff_180 = train_date - timedelta(days=180)
        cutoff_90 = train_date - timedelta(days=90)
        cutoff_30 = train_date - timedelta(days=30)
        
        recent_365 = hist[hist['date_arrival'] >= cutoff_365]
        recent_180 = hist[hist['date_arrival'] >= cutoff_180]
        recent_90 = hist[hist['date_arrival'] >= cutoff_90]
        recent_30 = hist[hist['date_arrival'] >= cutoff_30]
        
        # Basic aggregations
        if len(recent_365) > 0:
            total_365 = recent_365['net_weight'].sum()
            count_365 = len(recent_365)
            days_since = (train_date - recent_365['date_arrival'].max()).days
        else:
            total_365 = count_365 = days_since = 0
        
        if len(recent_180) > 0:
            total_180 = recent_180['net_weight'].sum()
            count_180 = len(recent_180)
        else:
            total_180 = count_180 = 0
        
        if len(recent_90) > 0:
            total_90 = recent_90['net_weight'].sum()
            count_90 = len(recent_90)
        else:
            total_90 = count_90 = 0
        
        if len(recent_30) > 0:
            total_30 = recent_30['net_weight'].sum()
            count_30 = len(recent_30)
            rate_30 = total_30 / 30
        else:
            total_30 = count_30 = rate_30 = 0
        
        # Rates
        rate_90 = total_90 / 90 if total_90 > 0 else 0
        
        # Recency-weighted sum
        if len(recent_90) > 0:
            days_ago = (train_date - recent_90['date_arrival']).dt.days
            weights = 1.0 / (days_ago + 1)
            recency_weighted = (recent_90['net_weight'] * weights).sum()
        else:
            recency_weighted = 0
        
        # Active days ratio
        if len(recent_90) > 0:
            active_days_90 = recent_90['date_arrival'].dt.date.nunique()
            active_ratio_90 = active_days_90 / 90
        else:
            active_ratio_90 = 0
        
        for horizon in forecast_horizons:
            forecast_end = train_date + timedelta(days=horizon)
            
            actual = receivals[
                (receivals['rm_id'] == rm_id) &
                (receivals['date_arrival'] >= train_date) &
                (receivals['date_arrival'] <= forecast_end)
            ]
            target = actual['net_weight'].sum()
            
            training_data.append({
                'rm_id': rm_id,
                'train_date': train_date,
                'forecast_horizon': horizon,
                'total_weight_365d': total_365,
                'count_365d': count_365,
                'days_since_last': days_since,
                'total_weight_90d': total_90,
                'count_90d': count_90,
                'rate_90': rate_90,
                'total_weight_180d': total_180,
                'count_180d': count_180,
                'total_30': total_30,
                'count_30': count_30,
                'rate_30': rate_30,
                'recency_weighted': recency_weighted,
                'active_ratio_90': active_ratio_90,
                'target': target
            })

print(f"\nGenerated {len(training_data)} training samples")
train_df = pd.DataFrame(training_data)

print(f"Samples with target > 0: {(train_df['target'] > 0).sum()} ({(train_df['target'] > 0).sum() / len(train_df) * 100:.1f}%)")

Using 11 training dates x 5 horizons
Active rm_ids in 2024: 60
Processing date 1/11: 2024-01-01...
Processing date 2/11: 2024-02-01...
Processing date 3/11: 2024-03-01...
Processing date 4/11: 2024-04-01...
Processing date 5/11: 2024-05-01...
Processing date 6/11: 2024-06-01...
Processing date 7/11: 2024-07-01...
Processing date 8/11: 2024-08-01...
Processing date 9/11: 2024-09-01...
Processing date 10/11: 2024-10-01...
Processing date 11/11: 2024-11-01...

Generated 2725 training samples
Samples with target > 0: 1816 (66.6%)


## Time-Based Train/Validation Split

In [5]:
split_date = pd.to_datetime('2024-09-01')

train_mask = train_df['train_date'] < split_date
val_mask = train_df['train_date'] >= split_date

feature_cols = [c for c in train_df.columns if c not in ['target', 'train_date']]

X_train = train_df[train_mask][feature_cols]
y_train = train_df[train_mask]['target']
X_val = train_df[val_mask][feature_cols]
y_val = train_df[val_mask]['target']

print(f"Training samples (before {split_date.date()}): {len(X_train)}")
print(f"Validation samples (>= {split_date.date()}): {len(X_val)}")

train_mean = y_train.mean()
val_mean = y_val.mean()
print(f"\nTarget statistics:")
print(f"  Training mean: {train_mean:,.0f} kg")
print(f"  Validation mean: {val_mean:,.0f} kg")
print(f"  Difference: {((val_mean - train_mean) / train_mean * 100):+.1f}%")

print(f"\nNumber of features: {len(feature_cols)}")

Training samples (before 2024-09-01): 1895
Validation samples (>= 2024-09-01): 830

Target statistics:
  Training mean: 341,946 kg
  Validation mean: 220,249 kg
  Difference: -35.6%

Number of features: 15


## Train LightGBM Models

Two-stage approach with classifier and quantile regressor (α=0.10).

In [6]:
# Classifier
y_train_bin = (y_train > 0).astype(int)
y_val_bin = (y_val > 0).astype(int)

clf = lgb.LGBMClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

print("Training LightGBM Classifier...")
clf.fit(
    X_train, y_train_bin,
    eval_set=[(X_train, y_train_bin), (X_val, y_val_bin)],
    eval_names=['training', 'valid_0'],
    callbacks=[lgb.log_evaluation(period=100)]
)

# Regressor with quantile objective
model = lgb.LGBMRegressor(
    objective='quantile',
    alpha=0.1,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

print(f"\nTraining LightGBM Regressor (quantile={model.alpha})...")
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['training', 'valid_0'],
    callbacks=[lgb.log_evaluation(period=100)]
)

Training LightGBM Classifier...
[100]	training's binary_logloss: 0.167808	valid_0's binary_logloss: 0.44933
[200]	training's binary_logloss: 0.0948759	valid_0's binary_logloss: 0.492214
[300]	training's binary_logloss: 0.0606071	valid_0's binary_logloss: 0.553723
[400]	training's binary_logloss: 0.0409862	valid_0's binary_logloss: 0.638244

Training LightGBM Regressor (quantile=0.1)...
[100]	training's quantile: 21104.8	valid_0's quantile: 22147.4
[200]	training's quantile: 15955.7	valid_0's quantile: 34537.2
[300]	training's quantile: 14043.2	valid_0's quantile: 41568.7


LGBMRegressor(alpha=0.1, colsample_bytree=0.8, learning_rate=0.05, max_depth=6,
              n_estimators=300, objective='quantile', random_state=42,
              subsample=0.8, verbose=-1)

## Feature Importance Analysis

In [7]:
# Classifier importance
clf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf.feature_importances_,
    'importance_normalized': clf.feature_importances_ / clf.feature_importances_.sum() * 100
}).sort_values('importance', ascending=False)

print("\nClassifier Feature Importance (Probability of Delivery)")
print("-" * 70)
print("Top features for predicting whether any delivery will occur:\n")
for idx, row in clf_importance.head(15).iterrows():
    print(f"  {row['feature']:25s}: {row['importance']:8.1f} ({row['importance_normalized']:5.2f}%)")

# Regressor importance
reg_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_,
    'importance_normalized': model.feature_importances_ / model.feature_importances_.sum() * 100
}).sort_values('importance', ascending=False)

print("\n\nRegressor Feature Importance (Quantile 0.10 Prediction)")
print("-" * 70)
print("Top features for predicting delivery weight:\n")
for idx, row in reg_importance.head(15).iterrows():
    print(f"  {row['feature']:25s}: {row['importance']:8.1f} ({row['importance_normalized']:5.2f}%)")

# Combined ranking
combined_importance = pd.merge(
    clf_importance[['feature', 'importance']],
    reg_importance[['feature', 'importance']],
    on='feature',
    suffixes=('_clf', '_reg')
)
combined_importance['avg_importance'] = (combined_importance['importance_clf'] + combined_importance['importance_reg']) / 2
combined_importance = combined_importance.sort_values('avg_importance', ascending=False)

print("\n\nCombined Feature Ranking (Average of Both Models)")
print("-" * 70)
print("Overall most important features:\n")
for idx, row in combined_importance.head(15).iterrows():
    print(f"  {row['feature']:25s}: Clf={row['importance_clf']:7.1f} | Reg={row['importance_reg']:7.1f} | Avg={row['avg_importance']:7.1f}")


Classifier Feature Importance (Probability of Delivery)
----------------------------------------------------------------------
Top features for predicting whether any delivery will occur:

  days_since_last          :   1446.0 (14.76%)
  total_weight_365d        :   1229.0 (12.55%)
  rm_id                    :   1143.0 (11.67%)
  recency_weighted         :    886.0 ( 9.04%)
  total_weight_180d        :    847.0 ( 8.65%)
  count_365d               :    788.0 ( 8.04%)
  forecast_horizon         :    709.0 ( 7.24%)
  total_weight_90d         :    639.0 ( 6.52%)
  total_30                 :    511.0 ( 5.22%)
  count_90d                :    433.0 ( 4.42%)
  count_180d               :    400.0 ( 4.08%)
  active_ratio_90          :    346.0 ( 3.53%)
  count_30                 :    176.0 ( 1.80%)
  rate_90                  :    150.0 ( 1.53%)
  rate_30                  :     93.0 ( 0.95%)


Regressor Feature Importance (Quantile 0.10 Prediction)
-----------------------------------------------

## Validation Performance Analysis

In [8]:
# Make predictions on validation set
val_reg_pred = np.maximum(0, model.predict(X_val))
val_clf_prob = clf.predict_proba(X_val)[:, 1]
val_combined_pred = val_reg_pred * val_clf_prob

# Create validation analysis dataframe
val_analysis = pd.DataFrame({
    'rm_id': X_val['rm_id'].values,
    'forecast_horizon': X_val['forecast_horizon'].values,
    'target': y_val.values,
    'reg_prediction': val_reg_pred,
    'clf_probability': val_clf_prob,
    'final_prediction': val_combined_pred
})

# Calculate quantile loss
def quantile_loss_0_2(y_true, y_pred):
    error = y_true - y_pred
    return np.where(error >= 0, 0.2 * error, -0.8 * error)

val_analysis['quantile_loss'] = quantile_loss_0_2(val_analysis['target'], val_analysis['final_prediction'])
val_analysis['abs_error'] = np.abs(val_analysis['target'] - val_analysis['final_prediction'])

print("\nOverall Validation Metrics")
print("-" * 70)
print(f"Total samples: {len(val_analysis)}")
print(f"Mean Quantile Loss (0.2): {val_analysis['quantile_loss'].mean():,.2f}")
print(f"Median Quantile Loss: {val_analysis['quantile_loss'].median():,.2f}")
print(f"Total Quantile Loss: {val_analysis['quantile_loss'].sum():,.2f}")
print(f"\nMean Absolute Error: {val_analysis['abs_error'].mean():,.2f}")
print(f"Mean Target: {val_analysis['target'].mean():,.2f}")
print(f"Mean Prediction: {val_analysis['final_prediction'].mean():,.2f}")

# Performance by horizon
print("\n\nPerformance by Forecast Horizon")
print("-" * 70)
print(f"{'Horizon':>8s} | {'Samples':>8s} | {'Avg Loss':>10s} | {'Avg Target':>12s} | {'Avg Pred':>12s} | {'Over/Under':>10s}")
print("-" * 70)
for horizon in sorted(val_analysis['forecast_horizon'].unique()):
    h_data = val_analysis[val_analysis['forecast_horizon'] == horizon]
    mean_loss = h_data['quantile_loss'].mean()
    mean_target = h_data['target'].mean()
    mean_pred = h_data['final_prediction'].mean()
    over_under = "OVER" if mean_pred > mean_target else "UNDER"
    pct_diff = ((mean_pred - mean_target) / mean_target * 100) if mean_target > 0 else 0
    print(f"{horizon:8d} | {len(h_data):8d} | {mean_loss:10,.2f} | {mean_target:12,.0f} | {mean_pred:12,.0f} | {over_under:>5s} {abs(pct_diff):4.1f}%")


Overall Validation Metrics
----------------------------------------------------------------------
Total samples: 830
Mean Quantile Loss (0.2): 45,847.23
Median Quantile Loss: 3,623.05
Total Quantile Loss: 38,053,204.46

Mean Absolute Error: 119,613.81
Mean Target: 220,248.89
Mean Prediction: 173,716.66


Performance by Forecast Horizon
----------------------------------------------------------------------
 Horizon |  Samples |   Avg Loss |   Avg Target |     Avg Pred | Over/Under
----------------------------------------------------------------------
       7 |      166 |   5,678.40 |       28,730 |       17,873 | UNDER 37.8%
      30 |      166 |  18,057.24 |      138,325 |       75,636 | UNDER 45.3%
      60 |      166 |  35,456.75 |      256,431 |      162,451 | UNDER 36.6%
      90 |      166 |  60,187.42 |      325,751 |      248,012 | UNDER 23.9%
     150 |      166 | 109,856.36 |      352,008 |      364,611 |  OVER  3.6%


## Pre-compute Features for 2025 Predictions

In [9]:
forecast_start = pd.to_datetime('2025-01-01')
rm_features = {}

for rm_id in prediction_mapping['rm_id'].unique():
    hist = receivals[
        (receivals['rm_id'] == rm_id) &
        (receivals['date_arrival'] < forecast_start)
    ]
    
    if len(hist) == 0:
        rm_features[rm_id] = {
            'total_weight_365d': 0,
            'count_365d': 0,
            'days_since_last': 999,
            'total_weight_90d': 0,
            'count_90d': 0,
            'rate_90': 0,
            'total_weight_180d': 0,
            'count_180d': 0,
            'total_30': 0,
            'count_30': 0,
            'rate_30': 0,
            'recency_weighted': 0,
            'active_ratio_90': 0
        }
        continue
    
    cutoff_365 = forecast_start - timedelta(days=365)
    cutoff_180 = forecast_start - timedelta(days=180)
    cutoff_90 = forecast_start - timedelta(days=90)
    cutoff_30 = forecast_start - timedelta(days=30)
    
    recent_365 = hist[hist['date_arrival'] >= cutoff_365]
    recent_180 = hist[hist['date_arrival'] >= cutoff_180]
    recent_90 = hist[hist['date_arrival'] >= cutoff_90]
    recent_30 = hist[hist['date_arrival'] >= cutoff_30]
    
    if len(recent_365) > 0:
        total_365 = recent_365['net_weight'].sum()
        count_365 = len(recent_365)
        days_since = (forecast_start - recent_365['date_arrival'].max()).days
    else:
        total_365 = count_365 = days_since = 0
    
    if len(recent_180) > 0:
        total_180 = recent_180['net_weight'].sum()
        count_180 = len(recent_180)
    else:
        total_180 = count_180 = 0
    
    if len(recent_90) > 0:
        total_90 = recent_90['net_weight'].sum()
        count_90 = len(recent_90)
        rate_90 = total_90 / 90
    else:
        total_90 = count_90 = rate_90 = 0
    
    if len(recent_30) > 0:
        total_30 = recent_30['net_weight'].sum()
        count_30 = len(recent_30)
        rate_30 = total_30 / 30
    else:
        total_30 = count_30 = rate_30 = 0
    
    # Recency-weighted
    if len(recent_90) > 0:
        days_ago = (forecast_start - recent_90['date_arrival']).dt.days
        weights = 1.0 / (days_ago + 1)
        recency_weighted = (recent_90['net_weight'] * weights).sum()
    else:
        recency_weighted = 0
    
    # Active ratio
    if len(recent_90) > 0:
        active_days_90 = recent_90['date_arrival'].dt.date.nunique()
        active_ratio_90 = active_days_90 / 90
    else:
        active_ratio_90 = 0
    
    rm_features[rm_id] = {
        'total_weight_365d': total_365,
        'count_365d': count_365,
        'days_since_last': days_since,
        'total_weight_90d': total_90,
        'count_90d': count_90,
        'rate_90': rate_90,
        'total_weight_180d': total_180,
        'count_180d': count_180,
        'total_30': total_30,
        'count_30': count_30,
        'rate_30': rate_30,
        'recency_weighted': recency_weighted,
        'active_ratio_90': active_ratio_90
    }

print(f"Pre-computed features for {len(rm_features)} rm_ids")

Pre-computed features for 203 rm_ids


## Generate Final Predictions

Apply two-stage model with guardrails for inactive materials.

In [10]:
predictions = []

for idx, row in prediction_mapping.iterrows():
    rm_id = row['rm_id']
    forecast_end = row['forecast_end_date']
    horizon = (forecast_end - forecast_start).days + 1
    
    feat = rm_features[rm_id]
    
    feature_dict = {
        'rm_id': rm_id,
        'forecast_horizon': horizon,
        'total_weight_365d': feat['total_weight_365d'],
        'count_365d': feat['count_365d'],
        'days_since_last': feat['days_since_last'],
        'total_weight_90d': feat['total_weight_90d'],
        'count_90d': feat['count_90d'],
        'rate_90': feat['rate_90'],
        'total_weight_180d': feat['total_weight_180d'],
        'count_180d': feat['count_180d'],
        'total_30': feat['total_30'],
        'count_30': feat['count_30'],
        'rate_30': feat['rate_30'],
        'recency_weighted': feat['recency_weighted'],
        'active_ratio_90': feat['active_ratio_90']
    }
    
    feature_vector = pd.DataFrame([feature_dict])[feature_cols]
    
    # Two-stage prediction
    reg_pred = max(0, model.predict(feature_vector)[0])
    prob_pos = clf.predict_proba(feature_vector)[:, 1][0]
    pred = reg_pred * prob_pos
    
    # Guardrails
    days_inactive = feat['days_since_last']
    total_365 = feat['total_weight_365d']
    cap_upper = (total_365 / 365.0) * horizon * 1.5
    
    if days_inactive > 365:
        pred = 0.0
    elif 180 < days_inactive <= 365:
        cold_cap = 0.08 * total_365
        pred = min(pred, cold_cap)
    
    pred = max(0.0, min(pred, cap_upper))
    
    predictions.append({'ID': row['ID'], 'predicted_weight': pred})
    
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(prediction_mapping)}...")

predictions_df = pd.DataFrame(predictions)

print(f"\nPredictions mean: {predictions_df['predicted_weight'].mean():,.0f} kg")
print(f"\nPrediction statistics:")
print(predictions_df['predicted_weight'].describe())
print(f"\nPredictions > 0: {(predictions_df['predicted_weight'] > 0).sum()}")

Processed 5000/30450...
Processed 10000/30450...
Processed 15000/30450...
Processed 20000/30450...
Processed 25000/30450...
Processed 30000/30450...

Predictions mean: 48,431 kg

Prediction statistics:
count    3.045000e+04
mean     4.843092e+04
std      2.394196e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.784078e+06
Name: predicted_weight, dtype: float64

Predictions > 0: 7562


## Save Submission

In [11]:
submission = sample_submission.copy()
submission['predicted_weight'] = predictions_df['predicted_weight'].values
submission.to_csv('Short_notebook_1.csv', index=False)
print("Submission saved: Short_notebook_1.csv")

Submission saved: Short_notebook_1.csv
